<a href="https://colab.research.google.com/github/blacklack547-hash/game-playtime-dashboard/blob/main/game_playtime_dashboard_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 — Install Python dependencies
!pip -q install --upgrade pandas streamlit plotly


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.


In [2]:
# Cell 2 — Download/inspect the source dataset
import pandas as pd

CSV_URL = "https://raw.githubusercontent.com/blacklack547-hash/game-playtime-dashboard/main/hltb_dataset_normalized.zip"

source_df = pd.read_csv(CSV_URL)
print("Dataset loaded successfully.")
print("Rows:", len(source_df))
print("Columns:", source_df.columns.tolist())
display(source_df.head())


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Dataset loaded successfully.
Rows: 166754
Columns: ['id', 'name', 'type', 'platform', 'genres', 'developer', 'publisher', 'release_date', 'release_precision', 'release_year', 'release_month', 'release_day', 'main_story_polled', 'main_story', 'main_plus_sides_polled', 'main_plus_sides', 'completionist_polled', 'completionist', 'all_styles_polled', 'all_styles', 'single_player_polled', 'single_player', 'co_op_polled', 'co_op', 'versus_polled', 'versus', 'source_url', 'crawled_at']


,id,name,type,platform,genres,developer,publisher,release_date,release_precision,release_year,...,all_styles_polled,all_styles,single_player_polled,single_player,co_op_polled,co_op,versus_polled,versus,source_url,crawled_at
0,1,688(I) Hunter/Killer,game,PC,"Simulation, Strategy/Tactical",Sonalysts,Electronic Arts,1997-07-04,day,1997.0,...,15.0,15.93,8.0,10.88,NaN,NaN,NaN,NaN,https://howlongtobeat.com/game/1,2026-03-27T10:06:13Z
1,2,Beyond Good & Evil 2,game,NaN,"Action, Adventure",Ubisoft Montpellier,Ubisoft,0000-00-00,day,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://howlongtobeat.com/game/2,2026-03-27T10:06:14Z
2,18,'Splosion Man,game,Xbox 360,Action,Twisted Pixel Games,Microsoft Games Studios,2009-07-22,day,2009.0,...,62.0,8.76,32.0,7.80,1.0,9.0,NaN,NaN,https://howlongtobeat.com/game/18,2026-03-27T10:06:16Z
3,19,.hack//G.U. Vol. 1: Rebirth,game,"Nintendo Switch, PC, PlayStation 2, PlayStation 4","Action, Role-Playing",CyberConnect2,Bandai,2006-05-18,day,2006.0,...,181.0,23.41,76.0,18.52,NaN,NaN,NaN,NaN,https://howlongtobeat.com/game/19,2026-03-27T10:06:15Z
4,20,.hack//G.U. Vol. 2: Reminisce,game,"Nintendo Switch, PC, PlayStation 2, PlayStation 4","Action, Role-Playing",CyberConnect2,Bandai,2006-09-28,day,2006.0,...,88.0,26.62,41.0,19.82,NaN,NaN,NaN,NaN,https://howlongtobeat.com/game/20,2026-03-27T10:06:15Z


In [3]:
# Cell 3 — Create the ETL pipeline script
%%writefile etl_pipeline.py
from pathlib import Path
import sqlite3
import pandas as pd

BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "output"
DB_PATH = OUTPUT_DIR / "backlog.db"
CSV_URL = "https://raw.githubusercontent.com/blacklack547-hash/game-playtime-dashboard/main/hltb_dataset_normalized.zip"

def run_etl():
    print("=== STARTING HLTB PLAYTIME ETL PIPELINE ===")

    # EXTRACT
    df = pd.read_csv(CSV_URL)
    print(f"Extracted {len(df):,} rows from the GitHub dataset.")

    # TRANSFORM
    df.columns = (
        df.columns.astype(str)
        .str.replace(" ", "_", regex=False)
        .str.lower()
        .str.strip()
    )

    before = len(df)
    if "name" in df.columns:
        df = df.dropna(subset=["name"])
    df = df.drop_duplicates()
    print(f"Removed {before - len(df):,} duplicate/invalid rows.")

    playtime_cols = ["main_story", "main_plus_sides", "completionist", "all_styles"]
    for col in playtime_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            median_value = df[col].median()
            if pd.isna(median_value):
                median_value = 0
            df[col] = df[col].fillna(median_value)

    text_cols = ["name", "type", "platform", "genres", "release_date", "source_url"]
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str).str.strip()

    # LOAD — this is the one canonical database path used by app.py
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with sqlite3.connect(DB_PATH) as conn:
        df.to_sql("curated_games", conn, if_exists="replace", index=False)

        # Verify the write before declaring success
        count = pd.read_sql_query(
            "SELECT COUNT(*) AS n FROM curated_games", conn
        ).iloc[0]["n"]

    print(f"Loaded {count:,} records into: {DB_PATH}")
    print("ETL completed successfully.")
    return count

if __name__ == "__main__":
    run_etl()


Writing etl_pipeline.py


In [4]:
# Cell 4 — Execute the ETL pipeline
!python etl_pipeline.py


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
=== STARTING HLTB PLAYTIME ETL PIPELINE ===
Extracted 166,754 rows from the GitHub dataset.
Removed 0 duplicate/invalid rows.
Loaded 166,754 records into: /content/output/backlog.db
ETL completed successfully.


In [5]:
# Cell 5 — Verify the exact database file and table used by the dashboard
from pathlib import Path
import sqlite3
import pandas as pd

DB_PATH = Path.cwd() / "output" / "backlog.db"

print("Working directory:", Path.cwd())
print("Database path:", DB_PATH)
print("Database exists:", DB_PATH.exists())
print("Database size (bytes):", DB_PATH.stat().st_size if DB_PATH.exists() else "MISSING")

if not DB_PATH.exists():
    raise FileNotFoundError(f"ETL database not found: {DB_PATH}")

with sqlite3.connect(DB_PATH) as conn:
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table'", conn
    )
    print("\nTables:")
    display(tables)

    if "curated_games" not in tables["name"].tolist():
        raise RuntimeError("The curated_games table was not created. Re-run Cell 4.")

    count = pd.read_sql_query(
        "SELECT COUNT(*) AS total FROM curated_games", conn
    ).iloc[0]["total"]
    print(f"\nRows in curated_games: {count:,}")
    display(pd.read_sql_query("SELECT * FROM curated_games LIMIT 5", conn))

if count == 0:
    raise RuntimeError("curated_games exists but contains zero rows.")


Working directory: /content
Database path: /content/output/backlog.db
Database exists: True
Database size (bytes): 31776768

Tables:


,name
0,curated_games



Rows in curated_games: 166,754


,id,name,type,platform,genres,developer,publisher,release_date,release_precision,release_year,...,all_styles_polled,all_styles,single_player_polled,single_player,co_op_polled,co_op,versus_polled,versus,source_url,crawled_at
0,1,688(I) Hunter/Killer,game,PC,"Simulation, Strategy/Tactical",Sonalysts,Electronic Arts,1997-07-04,day,1997.0,...,15.0,15.93,8.0,10.88,NaN,NaN,None,None,https://howlongtobeat.com/game/1,2026-03-27T10:06:13Z
1,2,Beyond Good & Evil 2,game,,"Action, Adventure",Ubisoft Montpellier,Ubisoft,0000-00-00,day,0.0,...,NaN,4.00,NaN,NaN,NaN,NaN,None,None,https://howlongtobeat.com/game/2,2026-03-27T10:06:14Z
2,18,'Splosion Man,game,Xbox 360,Action,Twisted Pixel Games,Microsoft Games Studios,2009-07-22,day,2009.0,...,62.0,8.76,32.0,7.80,1.0,9.0,None,None,https://howlongtobeat.com/game/18,2026-03-27T10:06:16Z
3,19,.hack//G.U. Vol. 1: Rebirth,game,"Nintendo Switch, PC, PlayStation 2, PlayStation 4","Action, Role-Playing",CyberConnect2,Bandai,2006-05-18,day,2006.0,...,181.0,23.41,76.0,18.52,NaN,NaN,None,None,https://howlongtobeat.com/game/19,2026-03-27T10:06:15Z
4,20,.hack//G.U. Vol. 2: Reminisce,game,"Nintendo Switch, PC, PlayStation 2, PlayStation 4","Action, Role-Playing",CyberConnect2,Bandai,2006-09-28,day,2006.0,...,88.0,26.62,41.0,19.82,NaN,NaN,None,None,https://howlongtobeat.com/game/20,2026-03-27T10:06:15Z


In [6]:
# Cell 6 — Write the Streamlit dashboard
%%writefile app.py
from pathlib import Path
import sqlite3
import pandas as pd
import streamlit as st
import plotly.express as px

st.set_page_config(
    page_title="Game Playtime Analytics Dashboard",
    page_icon="🎮",
    layout="wide"
)

# IMPORTANT: Must match the ETL output path exactly.
DB_PATH = Path(__file__).resolve().parent / "output" / "backlog.db"
TABLE_NAME = "curated_games"

@st.cache_data
def load_optimized_data(db_path: str, modified_time: float):
    # modified_time is included so Streamlit cache refreshes after ETL reruns.
    with sqlite3.connect(db_path) as conn:
        return pd.read_sql_query(f"SELECT * FROM {TABLE_NAME}", conn)

st.title("🎮 Video Game Playtime Curation & Analytics Platform")
st.caption("Explore HowLongToBeat-style playtime estimates across the game catalog.")

if not DB_PATH.exists():
    st.error(f"Database not found: {DB_PATH}")
    st.info("Run the ETL pipeline cell first, then restart this dashboard.")
    st.stop()

if DB_PATH.stat().st_size == 0:
    st.error(f"The database file exists but is empty: {DB_PATH}")
    st.info("Re-run the ETL pipeline cell and verify its success message.")
    st.stop()

try:
    df = load_optimized_data(str(DB_PATH), DB_PATH.stat().st_mtime)
except Exception as exc:
    st.error(f"Could not read '{TABLE_NAME}' from the database.")
    st.exception(exc)
    st.stop()

if df.empty:
    st.error("The curated_games table contains zero rows.")
    st.info("Run the ETL pipeline again and check Cell 5.")
    st.stop()

# Ensure numeric playtime columns are numeric even if source values vary.
playtime_cols = ["main_story", "main_plus_sides", "completionist", "all_styles"]
for col in playtime_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

st.success(f"Loaded {len(df):,} games from {DB_PATH}")

# KPI cards
k1, k2, k3 = st.columns(3)
k1.metric("🎮 Total catalogued titles", f"{len(df):,}")

if "main_story" in df.columns:
    valid_main = df.loc[df["main_story"] > 0, "main_story"]
    avg_main = valid_main.mean() if not valid_main.empty else 0
    k2.metric("⏱️ Average main story", f"{avg_main:.1f} hours")
else:
    k2.metric("⏱️ Average main story", "N/A")

if "completionist" in df.columns:
    valid_comp = df.loc[df["completionist"] > 0, "completionist"]
    avg_comp = valid_comp.mean() if not valid_comp.empty else 0
    k3.metric("🏆 Average 100% run", f"{avg_comp:.1f} hours")
else:
    k3.metric("🏆 Average 100% run", "N/A")

st.divider()

# Sidebar filters
st.sidebar.header("🔎 Filter games")
filtered = df.copy()

if "name" in filtered.columns:
    search_text = st.sidebar.text_input("Search game title")
    if search_text:
        filtered = filtered[
            filtered["name"].astype(str).str.contains(search_text, case=False, na=False)
        ]

if "type" in filtered.columns:
    type_values = sorted(filtered["type"].dropna().astype(str).unique().tolist())
    selected_types = st.sidebar.multiselect("Game type", type_values, default=type_values)
    if selected_types:
        filtered = filtered[filtered["type"].astype(str).isin(selected_types)]

if "platform" in filtered.columns:
    platform_values = sorted(filtered["platform"].dropna().astype(str).unique().tolist())
    selected_platforms = st.sidebar.multiselect("Platform", platform_values)
    if selected_platforms:
        filtered = filtered[filtered["platform"].astype(str).isin(selected_platforms)]

st.subheader("📈 Playtime insights")
left, right = st.columns(2)

with left:
    if {"name", "completionist"}.issubset(filtered.columns):
        longest = filtered[filtered["completionist"] > 0].nlargest(10, "completionist")
        st.markdown("#### 🏆 Top 10 longest games (100% completion)")
        if not longest.empty:
            fig = px.bar(
                longest.sort_values("completionist"),
                x="completionist", y="name", orientation="h",
                labels={"completionist": "Hours", "name": "Game"}
            )
            st.plotly_chart(fig, use_container_width=True)
        else:
            st.info("No completionist playtime values available for this selection.")

with right:
    if {"name", "main_story"}.issubset(filtered.columns):
        shortest = filtered[filtered["main_story"] > 0].nsmallest(10, "main_story")
        st.markdown("#### ⚡ 10 shortest main stories")
        if not shortest.empty:
            fig = px.bar(
                shortest.sort_values("main_story", ascending=False),
                x="main_story", y="name", orientation="h",
                labels={"main_story": "Hours", "name": "Game"}
            )
            st.plotly_chart(fig, use_container_width=True)
        else:
            st.info("No main-story playtime values available for this selection.")

st.divider()
st.subheader("🗂️ Browse game catalog")

display_cols = [
    c for c in [
        "name", "type", "platform", "main_story",
        "main_plus_sides", "completionist", "all_styles", "release_date"
    ] if c in filtered.columns
]

if display_cols:
    st.caption(f"Showing {len(filtered):,} matching games.")
    st.dataframe(
        filtered[display_cols].head(500),
        use_container_width=True,
        hide_index=True
    )
else:
    st.warning("Expected game columns were not found. Available columns:")
    st.write(df.columns.tolist())

st.caption("The table above is limited to the first 500 matching rows for responsiveness.")


Writing app.py


In [7]:
# Cell 7 — Install Cloudflare Tunnel (one-time per runtime)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared


In [ ]:
# Cell 8 — Launch Streamlit and expose it with Cloudflare Tunnel
# Run this cell once. It will keep running while the dashboard is open.
!streamlit run app.py --server.port 8501 --server.headless true & ./cloudflared tunnel --url http://localhost:8501


2026-11-23T03:41:03Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-23T03:41:03Z INF Requesting new quick Tunnel on trycloudflare.com...


2026-11-23T03:41:07Z INF +--------------------------------------------------------------------------------------------+
2026-11-23T03:41:07Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-23T03:41:07Z INF |  https://enough-queue-mpegs-installations.trycloudfl